# Inside wet-gas water condensation (v0.6.1)

Partial H2O condensation from a wet gas flowing inside bare tubes. The example demonstrates local onset (`T_wall,min < T_dew < T_wall,mean`), wet-area heat/mass transfer, final gas composition, endpoint properties, gas-phase hydraulics, Simulation and Rating.

In [1]:
import sys
from pathlib import Path

repository_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'core').is_dir())
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from core.geometry.bundle import TubeBundle
from core.geometry.tube import BareTube
from core.models.bare_tube import BareTubeHeatExchanger
from core.models.simulation import HXSideInput
from core.models.heat_balance import BalanceSideSpec
from core.properties.gas_mixture import GasMixturePropertyProvider, GasMixtureSpec
from core.phase_change.capability import detect_phase_change_capability

tube = BareTube(D_i=0.022, D_o=0.025, length_total=2.8, length_effective=2.8, wall_k=50.0)
bundle = TubeBundle(tube=tube, n_rows=5, n_tubes_per_row=10, pitch_transverse=0.035, pitch_longitudinal=0.035, layout='staggered', n_passes_tube=2, flow_arrangement='counterflow')
hx = BareTubeHeatExchanger(bundle)

wet_gas = GasMixtureSpec(components={'N2': 0.7318181818, 'O2': 0.1045454545, 'CO2': 0.0836363636, 'H2O': 0.08}, basis='mole')
dry_gas = GasMixtureSpec(components={'N2': 0.79, 'O2': 0.21}, basis='mole')
inside_provider = GasMixturePropertyProvider(wet_gas)
outside_provider = GasMixturePropertyProvider(dry_gas)
inside = HXSideInput(provider=inside_provider, m_dot=1.0, T_in=360.0, p=101_325.0)
outside = HXSideInput(provider=outside_provider, m_dot=5.0, T_in=290.0, p=101_325.0)
capability = detect_phase_change_capability(inside_provider)
print('INPUT')
print('inside composition:', wet_gas.to_mole_fractions())
print('W_in [kg/kg dry carrier]:', capability.W_in)
print('inside m_dot/T_in/p:', inside.m_dot, inside.T_in, inside.p)
print('outside m_dot/T_in/p:', outside.m_dot, outside.T_in, outside.p)

INPUT
inside composition: {'Nitrogen': 0.7318181818731818, 'Oxygen': 0.10454545451045454, 'CarbonDioxide': 0.08363636360836364, 'Water': 0.080000000008}
W_in [kg/kg dry carrier]: 0.052356988589459294
inside m_dot/T_in/p: 1.0 360.0 101325.0
outside m_dot/T_in/p: 5.0 290.0 101325.0


In [2]:
simulation = hx.simulate(inside, outside)
pc = simulation.inside_phase_change
hyd = simulation.tube_side_hydraulic.tube_bundle
summary = {
    'T_dew_in [K]': pc.dew_point_in, 'T_dew_out [K]': pc.dew_point_out,
    'T_wall_inside_min [K]': pc.wall_temperature_min, 'T_wall_inside_mean [K]': pc.wall_temperature_mean,
    'T_wall_inside_max [K]': pc.wall_temperature_max, 'T_wall_inside_wet_mean [K]': pc.wall_temperature_wet_mean,
    'wet_surface_fraction [-]': pc.wet_surface_fraction, 'A_wet [m2]': pc.wet_area,
    'W_in [kg/kg]': pc.W_in, 'W_mid [kg/kg]': pc.W_mid, 'W_out [kg/kg]': pc.W_out,
    'm_dot_gas_in [kg/s]': pc.m_dot_gas_in, 'm_dot_gas_out [kg/s]': pc.m_dot_gas_out,
    'm_dot_condensate [kg/s]': pc.m_dot_condensate,
    'Q_sensible [W]': pc.Q_sensible, 'Q_latent [W]': pc.Q_latent, 'Q_total [W]': pc.Q_total,
    'alfa_inside_dry [W/m2K]': pc.alfa_dry, 'alfa_inside_effective [W/m2K]': pc.alfa_effective,
    'inside dp friction [Pa]': hyd.dp_friction, 'inside dp acceleration [Pa]': hyd.dp_acceleration, 'inside dp total [Pa]': hyd.dp_total,
}
print('SIMULATION RESULT')
for name, value in summary.items():
    print(f'{name:38s} {value:.8g}')
assert pc.wall_temperature_min < pc.dew_point_in < pc.wall_temperature_mean
assert pc.active and pc.wet_surface_fraction > 0 and pc.wall_temperature_wet_mean < pc.dew_point_in
assert pc.m_dot_condensate > 0 and pc.W_out < pc.W_in and pc.Q_latent > 0

SIMULATION RESULT
T_dew_in [K]                           314.90989
T_dew_out [K]                          314.51292
T_wall_inside_min [K]                  311.90265
T_wall_inside_mean [K]                 324.79516
T_wall_inside_max [K]                  337.68769
T_wall_inside_wet_mean [K]             313.30756
wet_surface_fraction [-]               0.10897173
A_wet [m2]                             1.054422
W_in [kg/kg]                           0.052356989
W_mid [kg/kg]                          0.051768789
W_out [kg/kg]                          0.05118059
m_dot_gas_in [kg/s]                    1
m_dot_gas_out [kg/s]                   0.99888213
m_dot_condensate [kg/s]                0.001117877
Q_sensible [W]                         41363.269
Q_latent [W]                           2689.1922
Q_total [W]                            44052.461
alfa_inside_dry [W/m2K]                277.99257
alfa_inside_effective [W/m2K]          296.066
inside dp friction [Pa]                23364.238
insi

In [3]:
points = (simulation.inside_properties_inlet, simulation.inside_properties_midpoint, simulation.inside_properties_outlet)
labels = ('inlet', 'midpoint', 'outlet')
W_values = (pc.W_in, pc.W_mid, pc.W_out)
print(f"{'point':10s} {'T [K]':>10s} {'W':>12s} {'m_dot gas':>12s} {'rho':>12s} {'cp':>12s} {'mu':>12s} {'k':>12s} {'Pr':>10s} {'v':>12s} {'Re':>12s}")
for label, point, W in zip(labels, points, W_values):
    m_dot_gas = point.mass_flux * hyd.flow_area_per_pass
    print(f'{label:10s} {point.T:10.3f} {W:12.7f} {m_dot_gas:12.7f} {point.rho:12.6g} {point.cp:12.6g} {point.mu:12.6g} {point.k:12.6g} {point.Pr:10.6g} {point.velocity:12.6g} {point.reynolds:12.6g}')
assert points[2].mass_flux < points[1].mass_flux < points[0].mass_flux
assert all(all(__import__('math').isfinite(x) for x in (p.T, p.rho, p.cp, p.mu, p.k, p.Pr, p.velocity, p.reynolds)) for p in points)

point           T [K]            W    m_dot gas          rho           cp           mu            k         Pr            v           Re
inlet         360.000    0.0523570    1.0000000      0.98087         1055  1.97167e-05    0.0289935   0.717439      107.279       117412
midpoint      340.172    0.0517688    0.9994411      1.03856      1050.98  1.88509e-05    0.0275914   0.718052      101.263       122736
outlet        320.345    0.0511806    0.9988821      1.10343       1047.2  1.79674e-05    0.0271133   0.693961      95.2561       128699


In [4]:
rating = hx.rate(
    BalanceSideSpec(provider=inside_provider, p=inside.p, m_dot=inside.m_dot, T_in=inside.T_in, T_out=simulation.T_out_inside),
    BalanceSideSpec(provider=outside_provider, p=outside.p, m_dot=outside.m_dot, T_in=outside.T_in, T_out=simulation.T_out_outside),
)
rpc = rating.inside_phase_change
print('RATING RESULT')
print('active/converged/iterations:', rpc.active, rpc.converged, rpc.iterations)
print('W_in/W_mid/W_out:', rpc.W_in, rpc.W_mid, rpc.W_out)
print('wet fraction / condensate [kg/s]:', rpc.wet_surface_fraction, rpc.m_dot_condensate)
print('Q sensible/latent/total [W]:', rpc.Q_sensible, rpc.Q_latent, rpc.Q_total)
print('alfa inside dry/effective [W/m2K]:', rpc.alfa_dry, rpc.alfa_effective)
print('UA required/actual [W/K]:', rating.UA_required, rating.UA_actual)
assert rpc.active and rpc.converged and 0 < rpc.wet_surface_fraction < 1
assert abs(rpc.energy_balance_error) < 1e-5

RATING RESULT
active/converged/iterations: True True 3
W_in/W_mid/W_out: 0.052356988589459294 0.05176992447536045 0.0511828603612616
wet fraction / condensate [kg/s]: 0.17489254930885337 0.001115712862582356
Q sensible/latent/total [W]: 41365.479757161396 2686.982834944989 44052.462592106385
alfa inside dry/effective [W/m2K]: 277.9974061265311 283.49248456195255
UA required/actual [W/K]: 1000.645664136904 947.079071121435
